In [1]:
import math
import numpy as np
from scipy.integrate import quad_vec
from scipy.special import roots_legendre
from scipy import optimize
from scipy.stats import norm

In [2]:
i = 1j # imaginary number i
pi = np.pi # constant pi

In [3]:
lb, ub = 0, 200 # lower bound and upper bound for Gauss Quadrature 
Q = 0.5*(ub - lb) # multiple for change of interval of Gauss Quadrature
P = 0.5*(ub + lb) # constant added for change of interval of Gauss Quadrature
numgrid = 64 # number of nodes and weights used for Gauss Quadrature
u, w = roots_legendre(numgrid)

In [4]:
# Market Parameters
S = 1.0
K = 1.10
r = 0.02
q = 0
T = 1
# Model Parameters: theta = (kappa, nu_bar, sigma, rho, nu_0)^T
theta = np.array([3.0, 0.10, 0.25, -0.8, 0.08])

In [5]:
# Function for integrands of Heston pricer
def Hes_Int(u, a, b, c, rho, v0, K, T, S, r):
    '''
    a, b, c = kappa, nu_bar, sigma
    '''
    csqr = c**2
    PQ_M, PQ_N = P + Q*u, P - Q*u

    iPQ_M = i*PQ_M
    iPQ_N = i*PQ_N
    _iPQ_M = i*(PQ_M - i)
    _iPQ_N = i*(PQ_N - i)

    h_M = (K**(-iPQ_M))/iPQ_M
    h_N = (K**(-iPQ_N))/iPQ_N
    
    x0 = np.log(S) + r*T

    # kes = a - i*c*rho*u
    tmp = c*rho
    kes_M1 = a - tmp*_iPQ_M
    kes_N1 = a - tmp*_iPQ_N
    kes_M2 = kes_M1 + tmp
    kes_N2 = kes_N1 + tmp

    # m = i*u + u**2
    m_M1 = _iPQ_M + (PQ_M-i)**2
    m_N1 = _iPQ_N + (PQ_N-i)**2
    m_M2 = iPQ_M + PQ_M**2
    m_N2 = iPQ_N + PQ_N**2

    # d = sqrt(kes**2 + m*c**2)
    d_M1 = np.sqrt(kes_M1**2 + m_M1*csqr)
    d_N1 = np.sqrt(kes_N1**2 + m_N1*csqr)
    d_M2 = np.sqrt(kes_M2**2 + m_M2*csqr)
    d_N2 = np.sqrt(kes_N2**2 + m_N2*csqr)

    # g = exp(-a*b*rho*T*u*i/c)
    tmp1 = -a*b*rho*T/c
    tmp = np.exp(tmp1)
    g_M2 = np.exp(tmp1*iPQ_M)
    g_N2 = np.exp(tmp1*iPQ_N)
    g_M1 = g_M2*tmp
    g_N1 = g_N2*tmp

    # cosh and sinh for A term in char function
    halft = 0.5*T
    alpha = d_M1*halft
    calp_M1 = np.cosh(alpha)
    salp_M1 = np.sinh(alpha)

    alpha = d_N1*halft
    calp_N1 = np.cosh(alpha)
    salp_N1 = np.sinh(alpha)

    alpha = d_M2*halft
    calp_M2 = np.cosh(alpha)
    salp_M2 = np.sinh(alpha)

    alpha = d_N2*halft
    calp_N2 = np.cosh(alpha)
    salp_N2 = np.sinh(alpha)

    # A2 = d*calp + kes*salp
    A2_M1 = d_M1*calp_M1 + kes_M1*salp_M1
    A2_N1 = d_N1*calp_N1 + kes_N1*salp_N1
    A2_M2 = d_M2*calp_M2 + kes_M2*salp_M2
    A2_N2 = d_N2*calp_N2 + kes_N2*salp_N2

    # A1 = m*salp
    A1_M1 = m_M1*salp_M1
    A1_N1 = m_N1*salp_N1
    A1_M2 = m_M2*salp_M2
    A1_N2 = m_N2*salp_N2

    # A = A1/A2
    A_M1 = A1_M1/A2_M1
    A_N1 = A1_N1/A2_N1
    A_M2 = A1_M2/A2_M2
    A_N2 = A1_N2/A2_N2

    # D = log(d) + (a - d)*(T/2) - log((d+kes)/2 + (d-kes)/2*exp(-d*T))
    tmp = 2*a*b/csqr
    D_M1 = np.log(d_M1) + (a - d_M1)*halft - np.log((d_M1 + kes_M1)*0.5 + (d_M1 - kes_M1)*0.5*np.exp(-d_M1*T))
    D_N1 = np.log(d_N1) + (a - d_N1)*halft - np.log((d_N1 + kes_N1)*0.5 + (d_N1 - kes_N1)*0.5*np.exp(-d_N1*T))
    D_M2 = np.log(d_M2) + (a - d_M2)*halft - np.log((d_M2 + kes_M2)*0.5 + (d_M2 - kes_M2)*0.5*np.exp(-d_M2*T))
    D_N2 = np.log(d_N2) + (a - d_N2)*halft - np.log((d_N2 + kes_N2)*0.5 + (d_N2 - kes_N2)*0.5*np.exp(-d_N2*T))

    # char function: y1 = exp(i*x0*u) * exp(-v0*A) * g * exp(2*a*b/c**2 * D)
    MN_M1 = np.real(h_M*np.exp(x0*_iPQ_M - v0*A_M1 + tmp*D_M1)*g_M1)
    MN_N1 = np.real(h_N*np.exp(x0*_iPQ_N - v0*A_N1 + tmp*D_N1)*g_N1)
    MN_M2 = np.real(h_M*np.exp(x0*iPQ_M - v0*A_M2 + tmp*D_M2)*g_M2)
    MN_N2 = np.real(h_N*np.exp(x0*iPQ_N - v0*A_N2 + tmp*D_N2)*g_N2)

    MN = np.array([MN_M1, MN_N1, MN_M2, MN_N2])
    return MN


In [6]:
# Heston Pricer Function C(theta ; K, T)
def fHes(theta, K, T):
    '''
    
    '''
    # retrieve model parameters
    a, b, c, rho, v0 = theta[0], theta[1], theta[2], theta[3], theta[4]

    # numerical integration settings
    num_grid = int(np.ceil(numgrid/2))

    # calculate numerical integral
    disc = np.exp(-r*T)
    tmp = 0.5*(S - K*disc)
    discpi = disc/pi
    Y1, Y2 = 0.0, 0.0
    for j in range(num_grid):
        MN = Hes_Int(u[j], a, b, c, rho, v0, K, T, S, r)
        M1, N1, M2, N2 = MN[0], MN[1], MN[2], MN[3]

        Y1 += w[j]*(M1+N1)
        Y2 += w[j]*(M2+N2)

    QY1, QY2 = Q*Y1, Q*Y2
    price = tmp + discpi*(QY1 - K*QY2)
    return price

# For least_squares, residual vector is needed. Calculate vector of price - market_price

In [7]:
strikes = [0.3, 0.5, 0.7, 0.9, 1.00, 1.10, 1.30, 1.50]
for item in strikes:
    price = fHes(theta, item, T)
    print(price)

0.705976925383954
0.511742712682471
0.3305181080558398
0.18401726351134998
0.12907852300458011
0.08676616539666315
0.03449227757942344
0.011600982092717649


In [8]:
def heston_price_surface(maturities, strikes, theta):
    surface = []
    for t in maturities:
        temp_prices = []
        surface.append(temp_prices)
        for k in strikes:
            price = fHes(theta, k, t)
            temp_prices.append(price)
    surface = np.array(surface)
    return surface
            

In [9]:
maturities = [30/252, 60/252, 90/252, 120/252, 150/252, 180/252, 252/252, 360/252]
strikes = [0.3, 0.5, 0.7, 0.9, 1.00, 1.10, 1.30, 1.50]

In [10]:
mkt_prices = heston_price_surface(maturities, strikes, theta)
print(mkt_prices)
print(mkt_prices.shape)

[[7.00720416e-01 5.01189061e-01 3.01691239e-01 1.09909277e-01
  4.06797701e-02 8.36244082e-03 2.73541661e-05 1.19287261e-09]
 [7.01427382e-01 5.02376958e-01 3.03967151e-01 1.22987893e-01
  5.88331290e-02 2.13432172e-02 9.57385337e-04 7.47234872e-06]
 [7.02132759e-01 5.03587102e-01 3.07191548e-01 1.34775601e-01
  7.32843480e-02 3.35463623e-02 3.76892156e-03 1.61165456e-04]
 [7.02845408e-01 5.04867821e-01 3.11061669e-01 1.45444955e-01
  8.57716513e-02 4.48446048e-02 8.10750278e-03 7.93556948e-04]
 [7.03551592e-01 5.06251998e-01 3.15292312e-01 1.55239517e-01
  9.69663788e-02 5.53640822e-02 1.34222015e-02 2.13062666e-03]
 [7.04258672e-01 5.07747767e-01 3.19706763e-01 1.64336783e-01
  1.07216301e-01 6.52268818e-02 1.93158341e-02 4.19995836e-03]
 [7.05976925e-01 5.11742713e-01 3.30518108e-01 1.84017264e-01
  1.29078523e-01 8.67661654e-02 3.44922776e-02 1.16009821e-02]
 [7.08629288e-01 5.18489931e-01 3.46434896e-01 2.09555604e-01
  1.57054691e-01 1.14946265e-01 5.75706082e-02 2.65555117e-02]]

In [11]:
# Residual Vector Function
def HesPriceResid(theta, strikes, maturities, mkt_prices):
    '''
    Calculates the residual vector with each residual r(theta)_i = C(theta, K_i, T_i) - C*(K_i, T_i) where C* is the market price of a vanilla call option

    theta (5x1 array): Heston model parameters
    strikes (nx1 array): array of strikes (K_i)
    maturities (mx1 array): array of option maturities (T_i). Associated strikes and maturies make the option price surface
    mkt_prices (mxn array): array of market prices C*() with associated strikes and maturities K_i and T_i

    Requirement:
    - associated strikes and maturities with the respected market price must be in the same row and column for inputs into the function
    '''
    m = len(maturities)
    n = len(strikes)
    surface = heston_price_surface(maturities, strikes, theta)
    # matrix subtraction for price surface and market prices
    residuals = surface - mkt_prices
    # flatten matrix into 1d vector
    resid_vector = residuals.ravel()
    return resid_vector

In [12]:
def HesObjective(theta, strikes, maturities, mkt_prices):
    '''
    Takes residual vector from residual function and turns it into a single scalar objective function value
    Loss function chosen is least squares
    Used for optimize.minimize()
    '''
    residuals = HesPriceResid(theta, strikes, maturities, mkt_prices)
    return np.sum(residuals**2)

In [13]:
other = np.array([ 1.196e+00,  1.165e-01,  3.020e-01, -5.997e-01,  8.204e-02])

In [14]:
# Check if two different parameter sets produce the same residual vector
one = HesObjective(theta, strikes, maturities, mkt_prices)
two = HesObjective(theta, strikes, maturities, mkt_prices)
print(f'{one}, {two}')

0.0, 0.0


In [15]:
# Heston Analytical Gradient (Jacobian)
# Jacobian Integrands
def HesIntJac(u, a, b, c, rho, v0, K, T, S, r):
    '''

    '''
    # Put char function components here
    csqr = c**2
    PQ_M, PQ_N = P + Q*u, P - Q*u

    iPQ_M = i*PQ_M
    iPQ_N = i*PQ_N
    _iPQ_M = i*(PQ_M - i)
    _iPQ_N = i*(PQ_N - i)

    h_M = (K**(-iPQ_M))/iPQ_M
    h_N = (K**(-iPQ_N))/iPQ_N
    
    x0 = np.log(S) + r*T

    # kes = a - i*c*rho*u
    tmp = c*rho
    kes_M1 = a - tmp*_iPQ_M
    kes_N1 = a - tmp*_iPQ_N
    kes_M2 = kes_M1 + tmp
    kes_N2 = kes_N1 + tmp

    # m = i*u + u**2
    _msqr = (PQ_M - i)**2
    _nsqr = (PQ_N - i)**2
    msqr = PQ_M**2
    nsqr = PQ_N**2
    
    m_M1 = _iPQ_M + _msqr
    m_N1 = _iPQ_N + _nsqr
    m_M2 = iPQ_M + msqr
    m_N2 = iPQ_N + nsqr

    # d = sqrt(kes**2 + m*c**2)
    d_M1 = np.sqrt(kes_M1**2 + m_M1*csqr)
    d_N1 = np.sqrt(kes_N1**2 + m_N1*csqr)
    d_M2 = np.sqrt(kes_M2**2 + m_M2*csqr)
    d_N2 = np.sqrt(kes_N2**2 + m_N2*csqr)

    # g = exp(-a*b*rho*T*u*i/c
    tmp1 = -a*b*rho*T/c
    tmp2 = np.exp(tmp1)
    g_M2 = np.exp(tmp1*iPQ_M)
    g_N2 = np.exp(tmp1*iPQ_N)
    g_M1 = g_M2*tmp2
    g_N1 = g_N2*tmp2

    # cosh and sinh for A term in char function
    halft = 0.5*T
    alpha = d_M1*halft
    calp_M1 = np.cosh(alpha)
    salp_M1 = np.sinh(alpha)

    alpha = d_N1*halft
    calp_N1 = np.cosh(alpha)
    salp_N1 = np.sinh(alpha)

    alpha = d_M2*halft
    calp_M2 = np.cosh(alpha)
    salp_M2 = np.sinh(alpha)

    alpha = d_N2*halft
    calp_N2 = np.cosh(alpha)
    salp_N2 = np.sinh(alpha)

    # A2 = d*calp + kes*salp
    A2_M1 = d_M1*calp_M1 + kes_M1*salp_M1
    A2_N1 = d_N1*calp_N1 + kes_N1*salp_N1
    A2_M2 = d_M2*calp_M2 + kes_M2*salp_M2
    A2_N2 = d_N2*calp_N2 + kes_N2*salp_N2

    # A1 = m*salp
    A1_M1 = m_M1*salp_M1
    A1_N1 = m_N1*salp_N1
    A1_M2 = m_M2*salp_M2
    A1_N2 = m_N2*salp_N2

    # A = A1/A2
    A_M1 = A1_M1/A2_M1
    A_N1 = A1_N1/A2_N1
    A_M2 = A1_M2/A2_M2
    A_N2 = A1_N2/A2_N2

    # B = d*exp(a*T/2)/A2
    tmp = np.exp(a*halft) # exp(a*T/2)
    B_M1 = d_M1*tmp/A2_M1
    B_N1 = d_N1*tmp/A2_N1
    B_M2 = d_M2*tmp/A2_M2
    B_N2 = d_N2*tmp/A2_N2

    # D = log(d) + (a - d)*(T/2) - log((d+kes)/2 + (d-kes)/2*exp(-d*T))
    D_M1 = np.log(d_M1) + (a - d_M1)*halft - np.log((d_M1 + kes_M1)*0.5 + (d_M1 - kes_M1)*0.5*np.exp(-d_M1*T))
    D_N1 = np.log(d_N1) + (a - d_N1)*halft - np.log((d_N1 + kes_N1)*0.5 + (d_N1 - kes_N1)*0.5*np.exp(-d_N1*T))
    D_M2 = np.log(d_M2) + (a - d_M2)*halft - np.log((d_M2 + kes_M2)*0.5 + (d_M2 - kes_M2)*0.5*np.exp(-d_M2*T))
    D_N2 = np.log(d_N2) + (a - d_N2)*halft - np.log((d_N2 + kes_N2)*0.5 + (d_N2 - kes_N2)*0.5*np.exp(-d_N2*T))


    # Char Function: y1 = exp(i*x0*u) * exp(-v0*A) * g * exp(2*a*b/c**2 * D)
    tmp3 = 2*a*b/csqr
    y1_M1 = np.exp(x0*_iPQ_M - v0*A_M1 + tmp3*D_M1) * g_M1
    y1_N1 = np.exp(x0*_iPQ_N - v0*A_N1 + tmp3*D_N1) * g_N1
    y1_M2 = np.exp(x0*iPQ_M - v0*A_M2 + tmp3*D_M2) * g_M2
    y1_N2 = np.exp(x0*iPQ_N - v0*A_N2 + tmp3*D_N2) * g_N2

    # H = kes*calp + d*salp
    H_M1 = kes_M1*calp_M1 + d_M1*salp_M1
    H_N1 = kes_N1*calp_N1 + d_N1*salp_N1
    H_M2 = kes_M2*calp_M2 + d_M2*salp_M2
    H_N2 = kes_N2*calp_N2 + d_N2*salp_N2

    # ln(B) = D
    lnB_M1 = D_M1
    lnB_N1 = D_N1
    lnB_M2 = D_M2
    lnB_N2 = D_N2

    # partial b (nu_bar): y3 = y1*(2*a*lnB/c**2 - a*rho*T*u*i/c)
    tmp4 = tmp3/b
    tmp5 = tmp1/b

    y3_M1 = tmp4*lnB_M1 + tmp5*_iPQ_M
    y3_N1 = tmp4*lnB_N1 + tmp5*_iPQ_N
    y3_M2 = tmp4*lnB_M2 + tmp5*iPQ_M
    y3_N2 = tmp4*lnB_N2 + tmp5*iPQ_N

    # partial rho:
    tmp1 = tmp1/rho # -a*b*T/c

    # for M1:
    ctmp = c*_iPQ_M/d_M1
    pd_prho_M1 = -kes_M1*ctmp
    pA1_prho_M1 = m_M1*calp_M1*halft*pd_prho_M1
    pA2_prho_M1 = -ctmp*H_M1*(1+kes_M1*halft)
    pA_prho_M1 = (pA1_prho_M1 - A_M1*pA2_prho_M1)/A2_M1

    ctmp = pd_prho_M1 - pA2_prho_M1*d_M1/A2_M1
    pB_prho_M1 = tmp/A2_M1*ctmp # tmp = c*rho
    y4_M1 = -v0*pA_prho_M1 + tmp3*ctmp/d_M1 + tmp1*_iPQ_M

    # for N1:
    ctmp = c*_iPQ_N/d_N1
    pd_prho_N1 = -kes_N1*ctmp
    pA1_prho_N1 = m_N1*calp_N1*halft*pd_prho_N1
    pA2_prho_N1 = -ctmp*H_N1*(1+kes_N1*halft)
    pA_prho_N1 = (pA1_prho_N1 - A_N1*pA2_prho_N1)/A2_N1

    ctmp = pd_prho_N1 - pA2_prho_N1*d_N1/A2_N1
    pB_prho_N1 = tmp/A2_N1*ctmp
    y4_N1 = -v0*pA_prho_N1 + tmp3*ctmp/d_N1 + tmp1*_iPQ_N

    # for M2:
    ctmp = c*iPQ_M/d_M2
    pd_prho_M2 = -kes_M2*ctmp
    pA1_prho_M2 = m_M2*calp_M2*halft*pd_prho_M2
    pA2_prho_M2 = -ctmp*H_M2*(1+kes_M2*halft)
    pA_prho_M2 = (pA1_prho_M2 - A_M2*pA2_prho_M2)/A2_M2

    ctmp = pd_prho_M2 - pA2_prho_M2*d_M2/A2_M2
    pB_prho_M2 = tmp/A2_M2*ctmp
    y4_M2 = -v0*pA_prho_M2 + tmp3*ctmp/d_M2 + tmp1*iPQ_M

    # for N2:
    ctmp = c*iPQ_N/d_N2
    pd_prho_N2 = -kes_N2*ctmp
    pA1_prho_N2 = m_N2*calp_N2*halft*pd_prho_N2
    pA2_prho_N2 = -ctmp*H_N2*(1+kes_N2*halft)
    pA_prho_N2 = (pA1_prho_N2 - A_N2*pA2_prho_N2)/A2_N2

    ctmp = pd_prho_N2 - pA2_prho_N2*d_N2/A2_N2
    pB_prho_N2 = tmp/A2_N2*ctmp
    y4_N2 = -v0*pA_prho_N2 + tmp3*ctmp/d_N2 + tmp1*iPQ_N


    # partial a (kappa):
    tmp1 = b*rho*T/c
    tmp2 = tmp3/a # 2*b/c**2

    # for M1:
    ctmp = -1/(c*_iPQ_M)
    pB_pa = ctmp*pB_prho_M1 + B_M1*halft
    y5_M1 = -v0*pA_prho_M1*ctmp + tmp2*lnB_M1 + a*tmp2*pB_pa/B_M1 - tmp1*_iPQ_M

    # for N1:
    ctmp = -1/(c*_iPQ_N)
    pB_pa = ctmp*pB_prho_N1 + B_N1*halft
    y5_N1 = -v0*pA_prho_N1*ctmp + tmp2*lnB_N1 + a*tmp2*pB_pa/B_N1 - tmp1*_iPQ_N

    # for M2:
    ctmp = -1/(c*iPQ_M)
    pB_pa = ctmp*pB_prho_M2 + B_M2*halft
    y5_M2 = -v0*pA_prho_M2*ctmp + tmp2*lnB_M2 + a*tmp2*pB_pa/B_M2 - tmp1*iPQ_M

    # for N2:
    ctmp = -1/(c*iPQ_N)
    pB_pa = ctmp*pB_prho_N2 + B_N2*halft
    y5_N2 = -v0*pA_prho_N2*ctmp + tmp2*lnB_N2 + a*tmp2*pB_pa/B_N2 - tmp1*iPQ_N
    

    # partial c (sigma):
    tmp = rho/c
    tmp1 = 4*a*b/c**3
    tmp2 = a*b*rho*T/c**2

    # for M1:
    pd_pc = (tmp - 1/kes_M1)*pd_prho_M1 + c*_msqr/d_M1
    pA1_pc = m_M1*calp_M1*halft*pd_pc
    pA2_pc = tmp*pA2_prho_M1 - 1/(_iPQ_M)*(2/(T*kes_M1) + 1)*pA1_prho_M1 + c*halft*A1_M1
    pA_pc = pA1_pc/A2_M1 - A_M1/A2_M1*pA2_pc
    y6_M1 = -v0*pA_pc - tmp1*lnB_M1 + tmp3/d_M1*(pd_pc - d_M1/A2_M1*pA2_pc) + tmp2*_iPQ_M

    # for N1:
    pd_pc = (tmp - 1/kes_N1)*pd_prho_N1 + c*_nsqr/d_N1
    pA1_pc = m_N1*calp_N1*halft*pd_pc
    pA2_pc = tmp*pA2_prho_N1 - 1/(_iPQ_N)*(2/(T*kes_N1) + 1)*pA1_prho_N1 + c*halft*A1_N1
    pA_pc = pA1_pc/A2_N1 - A_N1/A2_N1*pA2_pc
    y6_N1 = -v0*pA_pc - tmp1*lnB_N1 + tmp3/d_N1*(pd_pc - d_N1/A2_N1*pA2_pc) + tmp2*_iPQ_N

    # for M2:
    pd_pc = (tmp - 1/kes_M2)*pd_prho_M2 + c*msqr/d_M2
    pA1_pc = m_M2*calp_M2*halft*pd_pc
    pA2_pc = tmp*pA2_prho_M2 - 1/(iPQ_M)*(2/(T*kes_M2) + 1)*pA1_prho_M2 + c*halft*A1_M2
    pA_pc = pA1_pc/A2_M2 - A_M2/A2_M2*pA2_pc
    y6_M2 = -v0*pA_pc - tmp1*lnB_M2 + tmp3/d_M2*(pd_pc - d_M2/A2_M2*pA2_pc) + tmp2*iPQ_M

    # for N2
    pd_pc = (tmp - 1/kes_N2)*pd_prho_N2 +  c*nsqr/d_N2
    pA1_pc = m_N2*calp_N2*halft*pd_pc
    pA2_pc = tmp*pA2_prho_N2 - 1/(iPQ_N)*(2/(T*kes_N2) + 1)*pA1_prho_N2 + c*halft*A1_N2
    pA_pc = pA1_pc/A2_N2 - A_N2/A2_N2*pA2_pc
    y6_N2 = -v0*pA_pc - tmp1*lnB_N2 + tmp3/d_N2*(pd_pc - d_N2/A2_N2*pA2_pc) + tmp2*iPQ_N


    # Char function with coefficient h = (K**(-iPQ))/iPQ for phi*plog(phi)/ptheta
    hM1 = h_M*y1_M1
    hM2 = h_M*y1_M2
    hN1 = h_N*y1_N1
    hN2 = h_N*y1_N2

    jac_pa1 = np.real(hM1*y5_M1 + hN1*y5_N1)
    jac_pa2 = np.real(hM2*y5_M2 + hN2*y5_N2)

    jac_pb1 = np.real(hM1*y3_M1 + hN1*y3_N1)
    jac_pb2 = np.real(hM2*y3_M2 + hN2*y3_N2)

    jac_pc1 = np.real(hM1*y6_M1 + hN1*y6_N1)
    jac_pc2 = np.real(hM2*y6_M2 + hN2*y6_N2)

    jac_prho1 = np.real(hM1*y4_M1 + hN1*y4_N1)
    jac_prho2 = np.real(hM2*y4_M2 + hN2*y4_N2)

    jac_pv01 = np.real(-hM1*A_M1 - hN1*A_N1) # partial v0: y2 = -A*y1
    jac_pv02 = np.real(-hM2*A_M2 - hN2*A_N2)

    jacobian = np.array([jac_pa1, jac_pa2, jac_pb1, jac_pb2, jac_pc1, jac_pc2, jac_prho1, jac_prho2, jac_pv01, jac_pv02])
    return jacobian


In [16]:
# return analytical gradient/jacobian of Heston Model
def JacHes(theta, K, T):
    '''
    returns 5x1 array
    '''
    # retrieve model parameters
    a, b, c, rho, v0 = theta[0], theta[1], theta[2], theta[3], theta[4]

    # numerical integration settings
    num_grid = int(np.ceil(numgrid/2))

    # calculate numerical integral
    discpi = np.exp(-r*T)/pi
    Y1, Y2 = 0.0, 0.0
    pa1, pa2 = 0.0, 0.0
    pb1, pb2 = 0.0, 0.0
    pc1, pc2 = 0.0, 0.0
    prho1, prho2 = 0.0, 0.0
    pv01, pv02 = 0.0, 0.0
    for j in range(num_grid):
        JI = HesIntJac(u[j], a, b, c, rho, v0, K, T, S, r) #JI = Jacobian Integral

        JI_pa1, JI_pa2 = JI[0], JI[1]
        JI_pb1, JI_pb2 = JI[2], JI[3]
        JI_pc1, JI_pc2 = JI[4], JI[5]
        JI_prho1, JI_prho2 = JI[6], JI[7]
        JI_pv01, JI_pv02 = JI[8], JI[9]

        pa1 += w[j]*JI_pa1
        pa2 += w[j]*JI_pa2

        pb1 += w[j]*JI_pb1
        pb2 += w[j]*JI_pb2

        pc1 += w[j]*JI_pc1
        pc2 += w[j]*JI_pc2

        prho1 += w[j]*JI_prho1
        prho2 += w[j]*JI_prho2

        pv01 += w[j]*JI_pv01
        pv02 += w[j]*JI_pv02

    jac = np.zeros(5)
    QY1, QY2 = Q*pa1, Q*pa2
    jac[0] = discpi*(QY1 - K*QY2)

    QY1, QY2 = Q*pb1, Q*pb2
    jac[1] = discpi*(QY1 - K*QY2)

    QY1, QY2 = Q*pc1, Q*pc2
    jac[2] = discpi*(QY1 - K*QY2)

    QY1, QY2 = Q*prho1, Q*prho2
    jac[3] = discpi*(QY1 - K*QY2)

    QY1, QY2 = Q*pv01, Q*pv02
    jac[4] = discpi*(QY1 - K*QY2)

    return jac

In [17]:
p = JacHes(theta, K, T)
print(p)
print(p.shape)

[ 0.00214018  0.45182082 -0.01984328  0.00492921  0.20376404]
(5,)


In [18]:
# Jacobian Matrix for multiple options for least_squares
def JacMatrix(theta, strikes, maturities, mkt_prices):
    matrix = []
    for t in maturities:
        for k in strikes:
            jac = JacHes(theta, k, t)
            matrix.append(jac)
    return np.array(matrix)

In [19]:
mat = JacMatrix(theta, strikes, maturities, mkt_prices)
print(mat.shape)
print(mat)

(64, 5)
[[-1.02831216e-06 -4.19202473e-05  7.49186785e-05 -3.68452874e-05
  -1.32816953e-04]
 [-6.88457850e-11  9.76528064e-09  1.51621441e-08 -3.22683142e-09
   6.21462353e-08]
 [-2.03775343e-07  2.08549371e-04  1.64296924e-04 -4.03999313e-05
   1.23042517e-03]
 [ 7.71400379e-05  1.94460681e-02  4.07869307e-03 -1.27124062e-03
   1.06492820e-01]
 [ 2.32228185e-04  3.78881735e-02 -1.03751413e-03  4.44653259e-05
   1.99774780e-01]
 [ 2.14277432e-04  2.65193813e-02 -6.14752355e-03  1.92266642e-03
   1.34588605e-01]
 [ 5.23553520e-06  4.72803993e-04 -2.49993420e-04  1.14236998e-04
   2.23170894e-03]
 [ 7.22193140e-10  5.99228311e-08 -3.84832111e-08  2.76318322e-08
   2.69319684e-07]
 [ 6.42949128e-07  8.36702142e-05 -1.41675324e-05 -1.15445158e-05
   2.08594437e-04]
 [-1.87432603e-07  2.73645144e-05  2.08637979e-05 -4.62101360e-06
   8.16630556e-05]
 [-5.70901533e-06  5.55655264e-03  2.10043071e-03 -5.41307848e-04
   1.53897014e-02]
 [ 2.30735807e-04  6.35342770e-02  5.77264025e-03 -2.0011

In [20]:
def HesGradient(theta, strikes, maturities, mkt_prices):
    r = HesPriceResid(theta, strikes, maturities, mkt_prices)  # (N,)
    J = JacMatrix(theta, strikes, maturities, mkt_prices)      # (N,5)

    return J.T @ r

In [21]:
print(HesGradient(theta, strikes, maturities, mkt_prices))

[0. 0. 0. 0. 0.]


In [22]:
''' 
Model Parameter Ranges:
- kappa: (0.50, 5.00)
- nu_bar: (0.05, 0.95)
- sigma: (0.05, 0.95)
- rho: (-0.90, -0.10)
- nu_0: (0.05, 0.95)

Paper uses initial guess (1.20, 0.20, 0.30, -0.60, 0.20)
'''
theta0 = np.array([1.20, 0.20, 0.30, -0.60, 0.20])
bounds = [
    (0.01, 20),      # kappa
    (0.001, 2),      # v_bar
    (0.01, 5),       # sigma
    (-0.999, 0.999), # rho
    (0.001, 2),      # v0
]

# ln transform to enforce positive values
'''
a = np.log(theta[0]) # kappa
b = np.log(theta[1]) # nu_bar
c = np.log(theta[2]) # sigma
d = np.tanh(theta[3]) # rho (to enforce -1 <= rho <= 1)
e = np.log(theta[4]) # nu_0
'''
# undo transforms with e^() and arctanh()

'\na = np.log(theta[0]) # kappa\nb = np.log(theta[1]) # nu_bar\nc = np.log(theta[2]) # sigma\nd = np.tanh(theta[3]) # rho (to enforce -1 <= rho <= 1)\ne = np.log(theta[4]) # nu_0\n'

In [23]:
# Feller Condition
def feller_cond(theta):
    kappa = theta[0]
    nu_bar = theta[1]
    sigma = theta[2]

    return 2*kappa*nu_bar - sigma**2

In [24]:
# Constraints
constraints = [
    {'type':'ineq',
     'fun':feller_cond}
]

In [25]:
'''
res.x          # calibrated parameters
res.cost       # 1/2 * sum(residuals**2)
res.fun        # residual vector
res.jac        # Jacobian at solution
res.success    # True/False
res.message    # termination reason
res.nfev       # number of residual evaluations
res.njev       # number of Jacobian evaluations
'''

'\nres.x          # calibrated parameters\nres.cost       # 1/2 * sum(residuals**2)\nres.fun        # residual vector\nres.jac        # Jacobian at solution\nres.success    # True/False\nres.message    # termination reason\nres.nfev       # number of residual evaluations\nres.njev       # number of Jacobian evaluations\n'

In [26]:
%%time
test = optimize.least_squares(
    fun = HesPriceResid,
    x0 = theta0,
    jac = JacMatrix,
    args = (strikes, maturities, mkt_prices),
    method = 'lm')

CPU times: total: 7.95 s
Wall time: 8.19 s


In [27]:
print(test)

     message: `xtol` termination condition is satisfied.
     success: True
      status: 3
         fun: [ 2.220e-16  8.882e-16 ...  1.027e-15  1.166e-15]
           x: [ 3.000e+00  1.000e-01  2.500e-01 -8.000e-01  8.000e-02]
        cost: 1.6961568723721167e-29
         jac: [[-1.028e-06 -4.192e-05 ... -3.685e-05 -1.328e-04]
               [-6.885e-11  9.765e-09 ... -3.227e-09  6.215e-08]
               ...
               [ 3.047e-03  5.470e-01 ...  1.020e-02  1.575e-01]
               [ 2.822e-03  4.059e-01 ...  1.135e-02  1.153e-01]]
        grad: [ 1.936e-17  2.266e-15 -2.910e-16  9.381e-17  8.453e-16]
  optimality: 2.2656975767650046e-15
 active_mask: [0 0 0 0 0]
        nfev: 12
        njev: 8


In [28]:
%%time
NM = optimize.minimize(
    fun = HesObjective,
    x0 = theta0,
    args = (strikes, maturities, mkt_prices),
    method = 'Nelder-Mead')

print(NM)

       message: Optimization terminated successfully.
       success: True
        status: 0
           fun: 7.308959888878393e-15
             x: [ 3.000e+00  1.000e-01  2.500e-01 -8.000e-01  8.000e-02]
           nit: 591
          nfev: 958
 final_simplex: (array([[ 3.000e+00,  1.000e-01, ..., -8.000e-01,
                         8.000e-02],
                       [ 3.000e+00,  1.000e-01, ..., -8.000e-01,
                         8.000e-02],
                       ...,
                       [ 3.000e+00,  1.000e-01, ..., -8.000e-01,
                         8.000e-02],
                       [ 3.000e+00,  1.000e-01, ..., -8.000e-01,
                         8.000e-02]]), array([ 7.309e-15,  1.528e-14,  1.542e-14,  1.733e-14,
                        1.841e-14,  2.544e-14]))
CPU times: total: 3min 55s
Wall time: 3min 59s


In [29]:
%%time
SLSQP = optimize.minimize(
    fun = HesObjective,
    x0 = theta0,
    args = (strikes, maturities, mkt_prices),
    method = 'SLSQP')

print(SLSQP)

 message: Optimization terminated successfully
 success: True
  status: 0
     fun: 3.7848187780345385e-05
       x: [ 1.196e+00  1.165e-01  3.020e-01 -5.997e-01  8.204e-02]
     nit: 6
     jac: [-7.866e-05 -6.077e-05  8.328e-04 -4.602e-04 -6.204e-04]
    nfev: 38
    njev: 6
CPU times: total: 9.47 s
Wall time: 9.54 s


In [30]:
%%time
LBFGSB = optimize.minimize(
    fun = HesObjective,
    x0 = theta0,
    #jac = HesGradient,
    args = (strikes, maturities, mkt_prices),
    method = 'L-BFGS-B',
    bounds = bounds)

print(LBFGSB)

  message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
  success: True
   status: 0
      fun: 6.576947726732835e-06
        x: [ 1.203e+00  1.107e-01  2.447e-01 -5.622e-01  8.242e-02]
      nit: 16
      jac: [-7.395e-06  3.399e-06  1.081e-05  2.001e-05  6.612e-06]
     nfev: 108
     njev: 18
 hess_inv: <5x5 LbfgsInvHessProduct with dtype=float64>
CPU times: total: 26.5 s
Wall time: 27 s


In [31]:
%%time
BFGS = optimize.minimize(
    fun = HesObjective,
    x0 = theta0,
    jac = HesGradient,
    args = (strikes, maturities, mkt_prices),
    method = 'BFGS')

print(BFGS)

  message: Optimization terminated successfully.
  success: True
   status: 0
      fun: 6.471304434911141e-06
        x: [ 1.204e+00  1.107e-01  2.429e-01 -5.661e-01  8.241e-02]
      nit: 16
      jac: [-3.684e-06  6.660e-07  4.939e-06  9.618e-06 -2.218e-06]
 hess_inv: [[ 1.925e+00 -8.460e-01 ...  5.779e+00 -7.898e-03]
            [-8.460e-01  3.684e+00 ... -5.648e+00 -1.659e+00]
            ...
            [ 5.779e+00 -5.648e+00 ...  3.722e+01  3.108e-01]
            [-7.898e-03 -1.659e+00 ...  3.108e-01  1.274e+00]]
     nfev: 18
     njev: 18
CPU times: total: 17.5 s
Wall time: 18 s
